# Lernmaterial-Generator – LangGraph-Agent für Notebook-Quizze

Dieses Notebook ist **selbst Lerngegenstand**: Es zeigt anhand eines praktischen
Anwendungsfalls, wie sich mit **LangGraph** ein Agenten-Workflow mit
*Reflexions-Schleife* aufbauen lässt.

**Was der Agent tut** – aus zwei Eingaben
1. einem hochgeladenen **Jupyter-Notebook** (Code *und* Markdown werden analysiert) und
2. einem **Schwerpunkt-Text** des Dozenten

erzeugt er drei aufeinander abgestimmte Artefakte:

1. ein **knappes, didaktisch sauberes Markdown-Skript** mit dem nötigen Theorie-Hintergrund,
2. eine **Liste der zugehörigen Fachtermini** und
3. eine **eigenständige HTML-Quiz-Seite** mit 10 Fragen im gewohnten Layout.

Ein **Qualitäts-/Reflexions-Knoten** prüft das Ergebnis (Halluzinationen, Wortzahl-Balance der
Antworten, Eindeutigkeit der Lösung, Schwerpunkt-Abdeckung) und löst bei Bedarf eine
Überarbeitung aus – **maximal drei Iterationen**.

Studierende können beliebig oft **neue Quizze** erzeugen (mit Schwierigkeitsstufe 1–5);
eine Anti-Dubletten-Logik verhindert Wiederholungen.

> **Modelle:** provider-agnostisch über die OpenAI-kompatible Schnittstelle –
> funktioniert mit **LM Studio** (lokal), **DeepInfra** und **OpenRouter**.
> Für die Generierung genügen drei Felder (`base_url`, `api_key`, `model`); für den
> Qualitäts-Checker lässt sich optional ein **eigenes (stärkeres) Modell** angeben.

> **Neu in dieser Version – modulare Struktur:** Die fachliche Prompt-Schicht (»der eigentliche Skill«) liegt jetzt im Paket `skills/`, die deterministischen Werkzeuge (LLM-Client, Notebook-Parser, HTML-Render, Qualitätschecks) im Paket `tools/`. Das Notebook selbst hält nur noch die **Orchestrierung**: State, Graph-Knoten, Graph-Aufbau und die Gradio-Oberfläche.

## Projektstruktur

Das Notebook ist die **Orchestrierungs- und Lern-Ebene**. Die wiederverwendbaren
Bausteine sind in zwei Pakete getrennt, die **direkt neben dem Notebook** liegen müssen
(gleiches Verzeichnis bzw. auf dem `sys.path`):

```
.
├── Lernmaterial_Generator_LangGraph_Version3.ipynb   ← State · Knoten · Graph · UI
├── skills/                  ← die fachliche Prompt-Schicht (»der Skill«)
│   ├── __init__.py          ·  bündelt die Exporte
│   ├── common.py            ·  GROUNDING_RULE, bloom_guidance (geteilte Bausteine)
│   ├── script.py            ·  build_script_prompt
│   ├── termini.py           ·  build_termini_prompt
│   ├── quiz.py              ·  build_quiz_prompt
│   └── checker.py           ·  build_checker_prompt
└── tools/                   ← deterministische Werkzeuge (kein LLM-»Urteil«)
    ├── __init__.py          ·  bündelt die Exporte
    ├── llm.py               ·  call_llm, extract_json, Token-Zähler (TOKENS)
    ├── notebook.py          ·  analyze_notebook
    ├── quiz_html.py         ·  QUIZ_TEMPLATE, build_quiz_html
    └── checks.py            ·  structural_checks, word_balance_check
```

**Trennlinie:** `skills/` enthält alles, was *Sprache an das Modell* richtet (die
fachliche Intelligenz in den Prompts). `tools/` enthält alles **Deterministische** –
reproduzierbare Funktionen ohne LLM-Bewertung. Der Token-Zähler reist mit dem
LLM-Client in `tools/llm.py`, da er technisch an jeden Aufruf gekoppelt ist.

In [ ]:
# Einmalig ausführen, falls die Pakete fehlen:
# %pip install langgraph langchain-core openai gradio nbformat

In [ ]:
# Standardbibliothek + Frameworks
import re
import random
import html as html_lib
import tempfile
from typing import TypedDict, List

import gradio as gr
from langgraph.graph import StateGraph, END

# --- Skills: die fachliche Prompt-Schicht -----------------------------------
from skills import (
    build_script_prompt,
    build_termini_prompt,
    build_quiz_prompt,
    build_checker_prompt,
)

# --- Tools: deterministische Werkzeuge + LLM-Client/Token-Zähler ------------
from tools import (
    TOKENS,
    reset_tokens,
    call_llm,
    extract_json,
    analyze_notebook,
    build_quiz_html,
    structural_checks,
    word_balance_check,
)

## Schritt 1 – Provider-agnostischer LLM-Client &nbsp;·&nbsp; `tools/llm.py`

LM Studio, DeepInfra und OpenRouter sprechen alle die **OpenAI-kompatible** API.
Deshalb genügt ein einziger Client, der über `base_url` umgeschaltet wird.

`call_llm()` ist die zentrale Aufruf-Funktion aller Graph-Knoten. Die Konfiguration
(`base_url`, `api_key`, `model`) reist im **State** mit – so bleiben die Knoten zustandslos
und gut testbar. Ein robuster JSON-Extraktor fängt typische Modell-Macken ab
(Code-Fences, Vor-/Nachtext).

> Der Code (`call_llm`, `extract_json`, Token-Zähler) ist nach `tools/llm.py` ausgelagert und oben bereits importiert.

## Schritt 2 – Notebook-Analyse &nbsp;·&nbsp; `tools/notebook.py`

Wir lesen mit `nbformat` **Code- und Markdown-Zellen** getrennt aus. Beides bildet
die *Faktenbasis* (das »Grounding«), auf die sich Skript und Quiz stützen dürfen.

> `analyze_notebook` liegt jetzt in `tools/notebook.py`.

## Schritt 3 – HTML-Template (eigenständig, ohne Server lauffähig) &nbsp;·&nbsp; `tools/quiz_html.py`

Das gewohnte Layout bleibt eine **feste Hülle**. Das Sprachmodell erzeugt ausschließlich
das `questions`-Array; wir spritzen es **inline** in die Seite ein.

Das ist bewusst so gelöst: Würde man die Fragen aus einer separaten `.json` per `fetch()`
nachladen, blockiert der Browser das beim Öffnen über `file://` (CORS / Local-File-Policy).
**Inline-Einbettung** umgeht das – die fertige `.html` lässt sich per Doppelklick von der
Festplatte öffnen, ganz ohne lokalen Server.

> `QUIZ_TEMPLATE` und `build_quiz_html` liegen jetzt in `tools/quiz_html.py`.

## Schritt 4 – Die Prompts / der »Skill« &nbsp;·&nbsp; `skills/`

Hier sitzt die fachliche Intelligenz. Die Prompts sind **domänen-unabhängig** formuliert –
sie funktionieren für PyTorch ebenso wie für sklearn, pandas, seaborn, Keras/CNN,
Transfer-Learning, ARIMA/Prophet, gensim oder NLTK.

**Zentrale Leitplanken**

- **Grounding gegen Halluzinationen:** Es darf nur behauptet werden, was (a) durch den
  Notebook-Code belegt oder (b) gesichertes, allgemein anerkanntes Fachwissen ist.
  *Keine* erfundenen API-Signaturen, Parameter, Default-Werte oder Zahlen.
- **Bloom-Mischung nach Schwierigkeit (1–5):** je höher die Stufe, desto stärker das Gewicht
  hochwertiger Taxonomiestufen.
- **Wortzahl-Balance:** alle vier Optionen ähnlich lang und gleich strukturiert – die
  richtige Antwort darf **nicht** die längste/detaillierteste sein (typischer LLM-Tell).
- **Anti-Dubletten:** bereits gestellte Themen/Fragestämme werden explizit ausgeschlossen.

> Jeder Prompt-Builder hat ein eigenes Untermodul (`skills/script.py`, `skills/termini.py`, `skills/quiz.py`, `skills/checker.py`); geteilte Bausteine (`GROUNDING_RULE`, `bloom_guidance`) stehen in `skills/common.py`. Alle vier `build_*_prompt`-Funktionen sind oben importiert.

## Schritt 5 – Der gemeinsame Zustand (State)

In LangGraph reichen die Knoten einen gemeinsamen `State` weiter. Jeder Knoten gibt ein
**Teil-Dictionary** zurück, das in den State gemischt wird.

In [ ]:
class QuizState(TypedDict, total=False):
    # Konfiguration / Eingaben
    base_url: str
    api_key: str
    model: str
    checker_base_url: str
    checker_api_key: str
    checker_model: str
    notebook_code: str
    notebook_markdown: str
    focus: str
    difficulty: int
    avoid_topics: List[str]
    quiz_focus_termini: List[str]
    quiz_temperature: float
    # Zwischen- und Endergebnisse
    script_md: str
    title: str
    termini: List[str]
    quiz_questions: List[dict]
    quiz_html: str
    quality: dict
    iteration: int

## Schritt 6 – Die Graph-Knoten

`script -> termini -> quiz -> quality`. Der Qualitäts-Knoten kombiniert **programmatische
Checks** (Wortzahl-Balance, Struktur – objektiv und billig) mit einer **LLM-Bewertung**
(Halluzinationen, Plausibilität – semantisch).

In [ ]:
def node_script(state: QuizState) -> dict:
    system, user = build_script_prompt(state)
    script = call_llm(state, system, user, temperature=0.25)
    m = re.search(r"^#\s+(.+)$", script, flags=re.MULTILINE)
    title = m.group(1).strip() if m else "Wissens-Check"
    return {"script_md": script, "title": title}


def node_termini(state: QuizState) -> dict:
    system, user = build_termini_prompt(state)
    raw = call_llm(state, system, user, temperature=0.2)
    try:
        termini = extract_json(raw)
        if not isinstance(termini, list):
            termini = [str(termini)]
    except ValueError:
        termini = [line.strip("-* ").strip() for line in raw.splitlines() if line.strip()]
    return {"termini": [str(t) for t in termini]}


def node_quiz(state: QuizState) -> dict:
    system, user = build_quiz_prompt(state)
    temp = state.get("quiz_temperature", 0.7)
    questions = []
    for _ in range(2):  # ein interner Wiederholversuch bei JSON-Fehler
        raw = call_llm(state, system, user, temperature=temp)
        try:
            questions = extract_json(raw)
            if isinstance(questions, list) and questions:
                break
        except ValueError:
            continue
    return {"quiz_questions": questions, "iteration": state.get("iteration", 0) + 1}


def node_quality(state: QuizState) -> dict:
    # 1) Programmatische Checks
    prog_issues = structural_checks(state["quiz_questions"]) + word_balance_check(state["quiz_questions"])
    # 2) LLM-Bewertung (faktentreu, niedrige Temperatur)
    system, user = build_checker_prompt(state)
    try:
        verdict = extract_json(call_llm(state, system, user, temperature=0.1, role="checker"))
    except ValueError:
        verdict = {"script_ok": True, "quiz_ok": True, "script_feedback": "", "quiz_feedback": ""}

    quiz_fb = (verdict.get("quiz_feedback") or "").strip()
    if prog_issues:
        quiz_fb = (quiz_fb + " | " if quiz_fb else "") + " ".join(prog_issues)
    quiz_ok = bool(verdict.get("quiz_ok", True)) and not prog_issues
    return {"quality": {
        "script_ok": bool(verdict.get("script_ok", True)),
        "quiz_ok": quiz_ok,
        "script_feedback": (verdict.get("script_feedback") or "").strip(),
        "quiz_feedback": quiz_fb,
    }}


def node_render(state: QuizState) -> dict:
    return {"quiz_html": build_quiz_html(state["quiz_questions"], state.get("title", "Wissens-Check"))}

## Schritt 7 – Programmatische Qualitätschecks &nbsp;·&nbsp; `tools/checks.py`

Objektive, billige Prüfungen vor dem LLM-Urteil. Besonders wichtig: der **Wortzahl-Bias**.
LLMs neigen dazu, die korrekte Antwort am längsten/ausführlichsten zu formulieren – wir
messen das und erzwingen bei systematischem Bias eine Überarbeitung.

> `structural_checks` und `word_balance_check` liegen jetzt in `tools/checks.py` und werden im Qualitäts-Knoten (Schritt 6) aufgerufen.

## Schritt 8 – Graph zusammenbauen und Routing

Zwei kompilierte Graphen aus denselben Knoten:

- **`full_graph`** – kompletter Lauf: `script -> termini -> quiz -> quality -> (render | zurück)`.
  Bei Skript-Mängeln geht es zurück zu `script` (Kaskade), bei reinen Quiz-Mängeln nur
  zu `quiz`. Nach **maximal 3 Iterationen** wird in jedem Fall gerendert (bestmögliches Ergebnis).
- **`quiz_graph`** – schlanker Pfad nur für neue Quizze: `quiz -> quality -> (render | quiz)`.
  Skript und Termini bleiben unangetastet.

In [ ]:
MAX_ITER = 3

def route_full(state: QuizState) -> str:
    q = state.get("quality", {})
    if q.get("script_ok", True) and q.get("quiz_ok", True):
        return "render"
    if state.get("iteration", 0) >= MAX_ITER:
        return "render"
    return "rewrite_script" if not q.get("script_ok", True) else "rewrite_quiz"


def route_quiz_only(state: QuizState) -> str:
    q = state.get("quality", {})
    if q.get("quiz_ok", True) or state.get("iteration", 0) >= MAX_ITER:
        return "render"
    return "rewrite_quiz"


def build_full_graph():
    g = StateGraph(QuizState)
    g.add_node("script", node_script)
    g.add_node("termini", node_termini)
    g.add_node("quiz", node_quiz)
    g.add_node("quality", node_quality)
    g.add_node("render", node_render)
    g.set_entry_point("script")
    g.add_edge("script", "termini")
    g.add_edge("termini", "quiz")
    g.add_edge("quiz", "quality")
    g.add_conditional_edges("quality", route_full,
                            {"render": "render", "rewrite_script": "script", "rewrite_quiz": "quiz"})
    g.add_edge("render", END)
    return g.compile()


def build_quiz_graph():
    g = StateGraph(QuizState)
    g.add_node("quiz", node_quiz)
    g.add_node("quality", node_quality)
    g.add_node("render", node_render)
    g.set_entry_point("quiz")
    g.add_edge("quiz", "quality")
    g.add_conditional_edges("quality", route_quiz_only,
                            {"render": "render", "rewrite_quiz": "quiz"})
    g.add_edge("render", END)
    return g.compile()


full_graph = build_full_graph()
quiz_graph = build_quiz_graph()

# Optionale Visualisierung des Graphen (benötigt zusätzliche Pakete):
try:
    from IPython.display import Image, display
    display(Image(full_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph-Visualisierung übersprungen:", e)
    print(full_graph.get_graph().draw_mermaid())

## Schritt 9 – Gradio-Oberfläche

Oben die Modell-Einstellungen, darunter die zwei Eingaben und die Schwierigkeitsstufe.
Der **vollständige Lauf** erzeugt Skript, Termini und Quiz inklusive **Download-Dateien**.

Drei Komfort-Funktionen sind eingebaut:

- **Fortschrittsanzeige:** Über `graph.stream(...)` wird nach *jedem* Knoten der Fortschritt
  gemeldet (»Schritt-Label · Tokens bisher«), inklusive Korrektur-Durchläufen der
  Reflexions-Schleife. So sieht man bei langen DeepInfra-Läufen, dass es vorangeht.
  *(Hinweis: Die Meldung erscheint jeweils nach Abschluss eines Knotens – innerhalb eines
  einzelnen LLM-Aufrufs gibt es keine Zwischenschritte.)*
- **Token-Verbrauch:** Jeder Aufruf summiert die `usage`-Felder der Modellantwort. Nach
  jedem Lauf werden Prompt-, Antwort- und Gesamt-Tokens angezeigt (sofern der Endpunkt
  `usage` liefert – LM Studio, DeepInfra und OpenRouter tun das). Optional lässt sich ein
  Mischpreis pro 1 Mio. Tokens eingeben, um eine grobe Kostenschätzung zu erhalten.
- **Abwechslung bei »Neues Quiz«:** Damit kleine lokale Modelle nicht immer dieselben Fragen
  liefern, wird bei jeder Neu-Generierung ein **rotierender, zufälliger Satz Fachtermini**
  als synthetischer Schwerpunkt in den Prompt injiziert (plus leicht erhöhte Temperatur und
  die Liste bereits gestellter Fragen als Ausschluss). Das erzwingt inhaltlich andere Fragen
  statt minimaler Umformulierungen.

In [ ]:
SESSION = {}  # hält das Ergebnis des letzten vollständigen Laufs (Skript, Termini, Notebook ...)

# ---- Hilfsfunktionen -------------------------------------------------------
def _write_tmp(text: str, suffix: str) -> str:
    f = tempfile.NamedTemporaryFile("w", suffix=suffix, delete=False, encoding="utf-8")
    f.write(text); f.close()
    return f.name

def _preview_iframe(html_str: str) -> str:
    safe = html_lib.escape(html_str, quote=True)
    return f'<iframe srcdoc="{safe}" style="width:100%;height:600px;border:1px solid #ddd;border-radius:8px;"></iframe>'

def _fmt(n: int) -> str:
    return f"{n:,}".replace(",", ".")   # deutsche Tausendertrennung

def _token_md(price_per_m: float = 0.0, spotlight=None) -> str:
    t = TOKENS
    s = (f"**Token-Verbrauch (dieser Lauf):** {_fmt(t['total'])} gesamt · "
         f"{_fmt(t['prompt'])} Prompt · {_fmt(t['completion'])} Antwort · {t['calls']} LLM-Aufrufe")
    if price_per_m and t["total"]:
        cost = t["total"] / 1_000_000 * price_per_m
        s += f"\n\n**Grobe Kostenschätzung:** ≈ {cost:.4f} € (Mischpreis {price_per_m:.2f} €/1 Mio. Tokens)"
    if spotlight:
        s += "\n\n**Synthetischer Schwerpunkt dieses Quiz:** " + ", ".join(spotlight)
    return s

# Nominale Knotenreihenfolge (für den Fortschrittsanteil) und Klartext-Labels
FULL_ORDER = ["script", "termini", "quiz", "quality", "render"]
REGEN_ORDER = ["quiz", "quality", "render"]
STEP_LABELS = {"script": "Skript schreiben", "termini": "Fachtermini extrahieren",
               "quiz": "Quiz generieren", "quality": "Qualitätsprüfung", "render": "Fertigstellen"}

def _stream_with_progress(graph, state, order, progress):
    """Führt den Graphen aus und meldet nach jedem Knoten den Fortschritt."""
    total = len(order)
    final = dict(state)
    done = 0
    progress(0.0, desc="Modell wird angesprochen …")
    for update in graph.stream(state, stream_mode="updates"):
        for node, delta in update.items():
            if delta:
                final.update(delta)
            done += 1
            it = final.get("iteration", 0)
            extra = f" · Korrektur-Durchlauf {it}" if node in ("quiz", "quality") and it > 1 else ""
            frac = min(done / max(total, 1), 0.99)
            progress(frac, desc=f"{STEP_LABELS.get(node, node)}{extra} · {_fmt(TOKENS['total'])} Tokens bisher")
    progress(1.0, desc=f"Fertig · {_fmt(TOKENS['total'])} Tokens")
    return final

def _select_focus_termini(k: int = 5):
    """Wählt rotierend einen zufälligen Satz Fachtermini als synthetischen Schwerpunkt."""
    termini = list(SESSION.get("termini", []))
    if not termini:
        return []
    used = SESSION.get("used_focus_termini", [])
    pool = [t for t in termini if t not in used]
    if len(pool) < min(k, len(termini)):   # fast alle verbraucht -> Rotation zurücksetzen
        used, pool = [], termini[:]
    random.shuffle(pool)
    chosen = pool[:k]
    SESSION["used_focus_termini"] = used + chosen
    return chosen

# ---- Haupt-Handler ---------------------------------------------------------
def run_full(base_url, api_key, model, checker_base_url, checker_api_key, checker_model,
             nb_file, focus, difficulty, price_per_m, progress=gr.Progress()):
    if not base_url or not model:
        raise gr.Error("Bitte base_url und model angeben.")
    if nb_file is None:
        raise gr.Error("Bitte ein Jupyter-Notebook (.ipynb) hochladen.")
    reset_tokens()
    parsed = analyze_notebook(nb_file.name)
    state = {"base_url": base_url, "api_key": api_key, "model": model,
             "checker_base_url": checker_base_url or "", "checker_api_key": checker_api_key or "",
             "checker_model": checker_model or "",
             "focus": focus or "", "difficulty": int(difficulty),
             "avoid_topics": [], "quiz_focus_termini": [], "quiz_temperature": 0.7,
             "iteration": 0, "quality": {}, **parsed}
    result = _stream_with_progress(full_graph, state, FULL_ORDER, progress)

    termini_md = "## Fachtermini\n\n" + "\n".join(f"- {t}" for t in result.get("termini", []))
    SESSION.clear()
    SESSION.update({k: result[k] for k in ("base_url","api_key","model",
                                           "checker_base_url","checker_api_key","checker_model",
                                           "focus","difficulty","notebook_code","notebook_markdown",
                                           "script_md","title","termini") if k in result})
    SESSION["avoid_topics"] = [q.get("text","") for q in result.get("quiz_questions", [])]
    SESSION["used_focus_termini"] = []

    price = float(price_per_m or 0)
    return (result.get("script_md",""), termini_md, _preview_iframe(result["quiz_html"]),
            _write_tmp(result.get("script_md",""), "_skript.md"),
            _write_tmp(termini_md, "_termini.md"),
            _write_tmp(result["quiz_html"], "_quiz.html"),
            _token_md(price))

def run_regen(difficulty, price_per_m, progress=gr.Progress()):
    if "script_md" not in SESSION:
        raise gr.Error("Bitte zuerst einen vollständigen Lauf ausführen.")
    reset_tokens()
    spotlight = _select_focus_termini(5)
    state = {**SESSION, "difficulty": int(difficulty), "iteration": 0, "quality": {},
             "quiz_focus_termini": spotlight, "quiz_temperature": 0.9}
    result = _stream_with_progress(quiz_graph, state, REGEN_ORDER, progress)
    SESSION["avoid_topics"] = SESSION.get("avoid_topics", []) + [q.get("text","") for q in result.get("quiz_questions", [])]
    price = float(price_per_m or 0)
    return _preview_iframe(result["quiz_html"]), _write_tmp(result["quiz_html"], "_quiz.html"), _token_md(price, spotlight)

# ---- Oberfläche ------------------------------------------------------------
with gr.Blocks(title="Lernmaterial-Generator") as demo:
    gr.Markdown("# Lernmaterial-Generator\nNotebook + Schwerpunkte → Skript, Fachtermini, Quiz.")
    with gr.Row():
        base_url = gr.Textbox(label="base_url", value="http://localhost:1234/v1",
                              placeholder="LM Studio: http://localhost:1234/v1 · OpenRouter: https://openrouter.ai/api/v1")
        api_key = gr.Textbox(label="api_key", type="password", placeholder="bei LM Studio leer lassen")
        model = gr.Textbox(label="model", placeholder="z. B. meta-llama-3.1-8b-instruct")
    with gr.Accordion("Checker-Modell (optional – leer lassen = gleiches Modell wie oben)", open=False):
        gr.Markdown("Für die Qualitäts-/Halluzinationsprüfung lohnt oft ein stärkeres Modell. "
                    "Leere Felder fallen automatisch auf das Generator-Modell oben zurück; "
                    "`base_url`/`api_key` dürfen abweichen (anderer Provider möglich).")
        with gr.Row():
            checker_base_url = gr.Textbox(label="checker base_url", placeholder="leer = wie oben")
            checker_api_key = gr.Textbox(label="checker api_key", type="password", placeholder="leer = wie oben")
            checker_model = gr.Textbox(label="checker model", placeholder="z. B. ein stärkeres Modell für die Prüfung")
    with gr.Row():
        nb_file = gr.File(label="Jupyter-Notebook (.ipynb)", file_types=[".ipynb"])
        with gr.Column():
            focus = gr.Textbox(label="Schwerpunkte des Dozenten", lines=4,
                               placeholder="z. B. Nimm besonderen Bezug auf die Wahl des Optimizers und stelle die drei wichtigsten Optimizer gegenüber.")
            difficulty = gr.Slider(1, 5, value=3, step=1, label="Schwierigkeitsstufe (1–5)")
            price_per_m = gr.Number(value=0, label="Preis €/1 Mio. Tokens (optional, grobe Schätzung)")
    run_btn = gr.Button("Lernmaterial erzeugen", variant="primary")
    token_box = gr.Markdown()

    with gr.Tab("Skript"):
        out_script = gr.Markdown()
    with gr.Tab("Fachtermini"):
        out_termini = gr.Markdown()
    with gr.Tab("Quiz"):
        out_quiz = gr.HTML()
        with gr.Row():
            regen_diff = gr.Slider(1, 5, value=3, step=1, label="Schwierigkeit für neues Quiz")
            regen_btn = gr.Button("Neues Quiz erzeugen")
    with gr.Row():
        dl_script = gr.File(label="Skript herunterladen")
        dl_termini = gr.File(label="Termini herunterladen")
        dl_quiz = gr.File(label="Quiz herunterladen")

    run_btn.click(run_full, [base_url, api_key, model, checker_base_url, checker_api_key, checker_model,
                             nb_file, focus, difficulty, price_per_m],
                  [out_script, out_termini, out_quiz, dl_script, dl_termini, dl_quiz, token_box])
    regen_btn.click(run_regen, [regen_diff, price_per_m], [out_quiz, dl_quiz, token_box])

demo.launch(inbrowser=True)